# Full Build Analysis - All Trace Files

This notebook analyzes **all 4,484 trace files** to understand the complete build time breakdown.

**Warning**: This processes ~46GB of data. We use streaming and aggregation to keep memory usage reasonable.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Add utils to path
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    utils_path = notebook_dir.parent / "utils"
else:
    utils_path = notebook_dir / "script" / "build_analysis" / "utils"

sys.path.insert(0, str(utils_path))

from trace_parser import (
    iter_trace_files,
    stream_events,
    microseconds_to_seconds,
)

print("✓ Imports successful")

## 1. Collect All Trace Files

In [ ]:
TRACE_DIR = Path.cwd().parent.parent.parent / "build-trace"

print(f"Scanning {TRACE_DIR}...")
trace_files = list(iter_trace_files(TRACE_DIR))
print(f"Found {len(trace_files):,} trace files")
print("\nFirst 5 files:")
for i, f in enumerate(trace_files[:5], 1):
    print(f"  {i}. {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)")

## 2. Aggregate Build Times Per File

This will take a few minutes as we process all files...

In [ ]:
file_stats = []

print("Processing all trace files...")
for trace_file in tqdm(trace_files, desc="Analyzing files"):
    try:
        total_duration = 0
        event_count = 0
        template_count = 0

        for event in stream_events(trace_file):
            event_count += 1
            total_duration += event.get("dur", 0)

            # Count template events
            if event.get("name") in ["InstantiateClass", "InstantiateFunction"]:
                template_count += 1

        file_stats.append(
            {
                "file": trace_file.name,
                "path": str(trace_file.relative_to(TRACE_DIR)),
                "total_duration_sec": microseconds_to_seconds(total_duration),
                "event_count": event_count,
                "template_count": template_count,
                "size_mb": trace_file.stat().st_size / (1024 * 1024),
            }
        )
    except Exception as e:
        print(f"Error processing {trace_file.name}: {e}")

print(f"\n✓ Processed {len(file_stats):,} files")

## 3. Overall Build Statistics

In [ ]:
df = pd.DataFrame(file_stats)

total_build_time = df["total_duration_sec"].sum()
total_events = df["event_count"].sum()
total_templates = df["template_count"].sum()

print("=" * 70)
print("FULL BUILD ANALYSIS")
print("=" * 70)
print(f"Total files analyzed:      {len(df):,}")
print(f"Total events:              {total_events:,}")
print(f"Total template events:     {total_templates:,}")
print(
    f"Total build time:          {total_build_time:.2f} seconds ({total_build_time / 60:.2f} minutes)"
)
print(f"Average time per file:     {df['total_duration_sec'].mean():.2f} seconds")
print(f"Median time per file:      {df['total_duration_sec'].median():.2f} seconds")
print("=" * 70)

## 4. Slowest Files to Compile

In [ ]:
slowest = df.nlargest(20, "total_duration_sec")

print("Top 20 slowest files to compile:\n")
for i, row in enumerate(slowest.itertuples(), 1):
    print(
        f"{i:2d}. {row.total_duration_sec:7.2f}s  {row.template_count:5,} templates  {row.file}"
    )

## 5. Save Results for Future Analysis

In [ ]:
# Save to CSV for faster re-analysis
output_file = Path.cwd().parent / "data" / "file_compilation_times.csv"
df.to_csv(output_file, index=False)
print(f"✓ Saved results to {output_file}")
print(f"  File size: {output_file.stat().st_size / 1024:.2f} KB")